# Basic EDA Implementation for Cleaned Parquet Files

This notebook starts implementation for a reusable basic EDA workflow over cleaned parquet datasets in `Data/Cleaned Data`.

Goal:
- Read all parquet files
- Run core checks (shape, columns, dtypes, missingness, duplicates)
- Produce a cross-file summary table

## 1. Section: Set Up Notebook Environment

Import required libraries, set display options, and define reusable constants for paths and runtime behavior.

In [7]:
import warnings
from pathlib import Path
import re
from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "Data" / "Cleaned Data"
PARQUET_PATTERNS = ("*.parquet", "*.parq", "*.pq", "*.parquest")
KEY_COLUMNS = ["Symbol", "Company Name", "Metrics"]

print(f"Base directory: {BASE_DIR}")
print(f"Data directory: {DATA_DIR}")

Base directory: h:\My Drive\Project\FA Project
Data directory: h:\My Drive\Project\FA Project\Data\Cleaned Data


## 2. Section: Define Project Configuration

Create a configuration object for data paths, feature flags, and defaults used in this implementation.

In [8]:
config = {
    "data_dir": DATA_DIR,
    "patterns": PARQUET_PATTERNS,
    "key_columns": KEY_COLUMNS,
    "show_top_n": 20,
    "read_validation": True,
}

print("Configuration loaded:")
for k, v in config.items():
    print(f"- {k}: {v}")

Configuration loaded:
- data_dir: h:\My Drive\Project\FA Project\Data\Cleaned Data
- patterns: ('*.parquet', '*.parq', '*.pq', '*.parquest')
- key_columns: ['Symbol', 'Company Name', 'Metrics']
- show_top_n: 20
- read_validation: True


## 3. Section: Implement Core Data Structures

Define typed structures representing per-file EDA outputs and the consolidated summary fields.

In [9]:
@dataclass
class EDAProfile:
    file: str
    path: str
    n_rows: int
    n_cols: int
    memory_mb: float
    n_missing_cells: int
    missing_pct_all_cells: float
    duplicate_rows: int
    n_numeric_cols: int
    has_symbol: bool
    has_company_name: bool
    has_metrics: bool
    n_quarter_cols: int
    quarter_start: str | None
    quarter_end: str | None


print("Core data structures ready.")

Core data structures ready.


## 4. Section: Implement Main Processing Functions

Implement file discovery, quarter-column detection, and the main parquet profiling function with modular helper outputs.

In [10]:
def discover_parquet_files(data_dir: Path, patterns: tuple[str, ...]) -> list[Path]:
    if not data_dir.exists():
        raise FileNotFoundError(f"Data directory not found: {data_dir.resolve()}")
    return sorted({p for pattern in patterns for p in data_dir.rglob(pattern)})


def get_quarter_cols(columns: list[str]) -> list[str]:
    return [c for c in columns if re.match(r"^CQ[1-4]\d{4}$", str(c))]


def profile_parquet(path: Path, key_cols: list[str]) -> tuple[EDAProfile, dict[str, Any]]:
    df = pd.read_parquet(path)
    n_rows, n_cols = df.shape
    n_missing_cells = int(df.isna().sum().sum())
    n_numeric_cols = int(len(df.select_dtypes(include=[np.number]).columns))

    dtype_table = (
        df.dtypes.astype(str).rename("dtype").to_frame()
        .assign(non_null=lambda x: [df[c].notna().sum() for c in x.index])
        .assign(nulls=lambda x: [df[c].isna().sum() for c in x.index])
        .assign(null_pct=lambda x: (x["nulls"] / n_rows * 100).round(2) if n_rows else 0.0)
        .reset_index().rename(columns={"index": "column"})
    )

    missing_table = (
        df.isna().sum().rename("missing_count").to_frame()
        .assign(missing_pct=lambda x: (x["missing_count"] / n_rows * 100).round(2) if n_rows else 0.0)
        .sort_values(["missing_pct", "missing_count"], ascending=False)
        .reset_index().rename(columns={"index": "column"})
    )

    key_diag_rows = []
    for col in key_cols:
        if col in df.columns:
            key_diag_rows.append(
                {
                    "column": col,
                    "null_count": int(df[col].isna().sum()),
                    "null_pct": round(df[col].isna().mean() * 100, 2),
                    "n_unique": int(df[col].nunique(dropna=True)),
                }
            )
    key_diag_df = pd.DataFrame(key_diag_rows)

    qcols = get_quarter_cols(list(df.columns))
    quarter_start = None
    quarter_end = None
    if qcols:
        q_sorted = sorted(qcols, key=lambda c: (int(str(c)[3:]), int(str(c)[2])))
        quarter_start, quarter_end = q_sorted[0], q_sorted[-1]

    profile = EDAProfile(
        file=path.name,
        path=str(path),
        n_rows=n_rows,
        n_cols=n_cols,
        memory_mb=round(df.memory_usage(deep=True).sum() / (1024 ** 2), 3),
        n_missing_cells=n_missing_cells,
        missing_pct_all_cells=round((n_missing_cells / (n_rows * n_cols) * 100), 4) if n_rows and n_cols else 0.0,
        duplicate_rows=int(df.duplicated().sum()),
        n_numeric_cols=n_numeric_cols,
        has_symbol="Symbol" in df.columns,
        has_company_name="Company Name" in df.columns,
        has_metrics="Metrics" in df.columns,
        n_quarter_cols=len(qcols),
        quarter_start=quarter_start,
        quarter_end=quarter_end,
    )

    details = {
        "columns": list(df.columns),
        "dtype_table": dtype_table,
        "missing_table": missing_table,
        "key_diagnostics": key_diag_df,
        "numeric_summary": df.describe(include=[np.number]).T if n_numeric_cols > 0 else pd.DataFrame(),
    }
    return profile, details


print("Processing functions ready.")

Processing functions ready.


## 5. Section: Run a Minimal End-to-End Workflow

Execute the workflow on all cleaned parquet files and print structured outputs for quick verification.

In [11]:
parquet_files = discover_parquet_files(config["data_dir"], config["patterns"])
if not parquet_files:
    raise FileNotFoundError("No parquet files found under Data/Cleaned Data.")

print(f"Discovered {len(parquet_files)} file(s):")
for p in parquet_files:
    print(f"- {p}")

profiles: dict[str, EDAProfile] = {}
profile_details: dict[str, dict[str, Any]] = {}

for p in parquet_files:
    print("\n" + "=" * 100)
    print(f"Profiling file: {p.name}")

    if config["read_validation"]:
        _ = pd.read_parquet(p)

    profile, details = profile_parquet(p, config["key_columns"])
    profiles[p.name] = profile
    profile_details[p.name] = details

    print(f"Shape: ({profile.n_rows}, {profile.n_cols}) | Memory(MB): {profile.memory_mb}")
    print(f"Duplicate rows: {profile.duplicate_rows}")
    print(f"Missing cells: {profile.n_missing_cells:,} ({profile.missing_pct_all_cells}%)")
    print(f"Quarter columns: {profile.n_quarter_cols}")
    if profile.quarter_start and profile.quarter_end:
        print(f"Quarter span: {profile.quarter_start} -> {profile.quarter_end}")

    print("\nColumns:")
    print(details["columns"])

    print("\nDtypes and null profile (top rows):")
    display(details["dtype_table"].head(config["show_top_n"]))

    print("\nMissingness profile (top rows):")
    display(details["missing_table"].head(config["show_top_n"]))

    if not details["key_diagnostics"].empty:
        print("\nKey-column checks:")
        display(details["key_diagnostics"])

    if not details["numeric_summary"].empty:
        print("\nNumeric summary (top rows):")
        display(details["numeric_summary"].head(config["show_top_n"]))

summary_df = pd.DataFrame([
    {
        "file": p.file,
        "rows": p.n_rows,
        "columns": p.n_cols,
        "memory_mb": p.memory_mb,
        "missing_cells": p.n_missing_cells,
        "missing_pct_all_cells": p.missing_pct_all_cells,
        "duplicate_rows": p.duplicate_rows,
        "numeric_columns": p.n_numeric_cols,
        "has_Symbol": p.has_symbol,
        "has_Company_Name": p.has_company_name,
        "has_Metrics": p.has_metrics,
        "quarter_columns": p.n_quarter_cols,
        "quarter_start": p.quarter_start,
        "quarter_end": p.quarter_end,
    }
    for p in profiles.values()
]).sort_values("file").reset_index(drop=True)

print("\nCross-file summary:")
display(summary_df)

Discovered 3 file(s):
- h:\My Drive\Project\FA Project\Data\Cleaned Data\clean_crsp.parquet
- h:\My Drive\Project\FA Project\Data\Cleaned Data\master_panel.parquet
- h:\My Drive\Project\FA Project\Data\Cleaned Data\transcript_quarter_panel.parquet

Profiling file: clean_crsp.parquet
Shape: (493203, 21) | Memory(MB): 92.445
Duplicate rows: 0
Missing cells: 17 (0.0002%)
Quarter columns: 0

Columns:
['permno', 'permco', 'ticker', 'cusip', 'issuernm', 'siccd', 'naics', 'dlyclose', 'dlyopen', 'dlyhigh', 'dlylow', 'dlyprc', 'dlyret', 'dlyretx', 'dlyvol', 'shrout', 'dlycap', 'sprtrn', 'vwretd', 'ewretd', 'date']

Dtypes and null profile (top rows):


,column,dtype,non_null,nulls,null_pct
0,permno,int64,493203,0,0.0
1,permco,int64,493203,0,0.0
2,ticker,str,493203,0,0.0
3,cusip,str,493203,0,0.0
4,issuernm,str,493203,0,0.0
5,siccd,int64,493203,0,0.0
6,naics,int64,493203,0,0.0
7,dlyclose,float64,493201,2,0.0
8,dlyopen,float64,493201,2,0.0
9,dlyhigh,float64,493201,2,0.0



Missingness profile (top rows):


,column,missing_count,missing_pct
0,dlyret,3,0.0
1,dlyretx,3,0.0
2,dlyclose,2,0.0
3,dlyopen,2,0.0
4,dlyhigh,2,0.0
5,dlylow,2,0.0
6,dlyprc,1,0.0
7,dlyvol,1,0.0
8,dlycap,1,0.0
9,permno,0,0.0



Numeric summary (top rows):


...

,column,missing_count,missing_pct
0,Numeric Transeprency,3,0.04
1,language_complexity,3,0.04
2,net_positivity,3,0.04
3,ticker,0,0.00
4,quarter,0,0.00
5,analyst_selectivity_ratio,0,0.00
6,quarter_end,0,0.00



Numeric summary (top rows):


,count,mean,std,min,25%,50%,75%,max
Numeric Transeprency,8117.0,2.328527,0.762707,0.29,1.79,2.25,2.78,7.46
analyst_selectivity_ratio,8120.0,41.818329,16.358235,0.00,30.00,40.00,52.94,100.00
language_complexity,8117.0,12.422687,1.136442,9.05,11.62,12.37,13.14,17.74
net_positivity,8117.0,1.105240,0.594777,-1.43,0.73,1.11,1.49,3.32



Cross-file summary:


,file,rows,columns,memory_mb,missing_cells,missing_pct_all_cells,duplicate_rows,numeric_columns,has_Symbol,has_Company_Name,has_Metrics,quarter_columns,quarter_start,quarter_end
0,clean_crsp.parquet,493203,21,92.445,17,0.0002,0,17,False,False,False,0,None,None
1,master_panel.parquet,479783,27,115.027,76116,0.5876,0,21,False,False,False,0,None,None
2,transcript_quarter_panel.parquet,8120,7,0.511,9,0.0158,0,4,False,False,False,0,None,None


## 6. Section: Add Basic Validation and Unit-Style Checks

Run assertion-based checks for expected files, non-empty outputs, and required EDA fields.

In [12]:
expected_files = {"clean_crsp.parquet", "master_panel.parquet", "transcript_quarter_panel.parquet"}
found_files = {p.name for p in parquet_files}

assert len(parquet_files) > 0, "No parquet files were discovered."
assert expected_files.issubset(found_files), (
    f"Missing expected cleaned files: {sorted(expected_files - found_files)}"
)
assert not summary_df.empty, "Cross-file summary is empty."
assert (summary_df["rows"] > 0).all(), "One or more files has zero rows."
assert (summary_df["columns"] > 0).all(), "One or more files has zero columns."

required_summary_cols = {
    "file", "rows", "columns", "missing_cells", "duplicate_rows", "numeric_columns"
}
assert required_summary_cols.issubset(set(summary_df.columns)), "Summary is missing required columns."

print("All basic validation checks passed.")

All basic validation checks passed.
